# Text-To-Brain Generation Comparison

Notebook wrapper for the MLP NeuroVLM vs atlas-free CNN Stage 4 text-to-brain generation comparison. The evaluation code lives in `atlas_free_cnn.evaluation.compare_text_to_brain_generation`; this notebook configures and calls the actual comparison workflow, then inspects the saved output tables.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neurovlm").exists() and (candidate / "experiments" / "3dcnn" / "atlas_free_cnn").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from the neurovlm repo or update this cell.")

REPO_ROOT = find_repo_root()
THREEDCNN = REPO_ROOT / "experiments" / "3dcnn"
MODEL_COMPARISON_DIR = THREEDCNN / "model_comparison"
for path in [REPO_ROOT / "src", THREEDCNN, MODEL_COMPARISON_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

REPO_ROOT

In [ ]:
import pandas as pd

from atlas_free_cnn.evaluation.compare_text_to_brain_generation import (
    BY_SAMPLE_FILENAME,
    DATASETS,
    DEFAULT_OUTPUT_DIR,
    DICE_SENSITIVITY_FILENAME,
    RANDOM_BASELINE_FILENAME,
    SUMMARY_FILENAME,
    T2B_MODEL_IDS,
    run_comparison,
)

list(DATASETS), list(T2B_MODEL_IDS)

## Configure

Set `RUN_MODE` below: `"quick"` is a fast sanity check (MLP on Nilearn only,
two samples, spin tests off); `"full"` is the actual comparison across all
datasets/models (downloads MLP assets, CNN Stage 4 checkpoints + matching
AEs + Stage 3 evaluators, the unified test split, and the normalized
SPECTER2 cache on first run). Everything below -- Run Comparison, Inspect
Outputs, Visualize Results -- uses whichever mode you pick here.

In [ ]:
# RUN_MODE:
#   "quick" - fast sanity check: MLP only, Nilearn only, two samples, spin tests off.
#   "full"  - the actual comparison across all datasets/models.
RUN_MODE = "full"

if RUN_MODE == "quick":
    DATASETS_TO_RUN = ["nilearn"]
    MODELS = ["mlp_neurovlm"]
    LIMIT = 2
    SKIP_SPIN = True
elif RUN_MODE == "full":
    DATASETS_TO_RUN = ["pubmed", "nilearn", "neurovault"]
    MODELS = [
        "mlp_neurovlm",
        "cnn_t2b_mixed",
        "cnn_t2b_pubmed",
        "cnn_t2b_nilearn",
        "cnn_t2b_neurovault",
    ]
    LIMIT = 8
    SKIP_SPIN = True
else:
    raise ValueError(f"Unknown RUN_MODE {RUN_MODE!r}; expected 'quick' or 'full'")

DEVICE = "cpu"
BATCH_SIZE = 8
SKIP_VOXEL_AUROC = False
DICE_PCT = 90.0
DICE_SENSITIVITY_PCTS = [80.0, 85.0, 90.0, 95.0]

preferred_test_jsonl = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "unified_jsonl" / "splits" / "test.jsonl"
TEST_JSONL = preferred_test_jsonl if preferred_test_jsonl.exists() else None

preferred_text_cache = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "text_embeddings" / "specter2_stage3_stage4_emptycentered_unitnorm.pt"
TEXT_EMBEDDING_CACHE = preferred_text_cache if preferred_text_cache.exists() else None

OUTPUT_DIR = REPO_ROOT / DEFAULT_OUTPUT_DIR

{
    "run_mode": RUN_MODE,
    "datasets": DATASETS_TO_RUN,
    "models": MODELS,
    "limit": LIMIT,
    "device": DEVICE,
    "skip_spin": SKIP_SPIN,
    "test_jsonl": str(TEST_JSONL) if TEST_JSONL else None,
    "text_embedding_cache": str(TEXT_EMBEDDING_CACHE) if TEXT_EMBEDDING_CACHE else None,
    "output_dir": str(OUTPUT_DIR),
}

## Run Comparison

In [ ]:
result = run_comparison(
    datasets=DATASETS_TO_RUN,
    models=MODELS,
    limit=LIMIT,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    test_jsonl=TEST_JSONL,
    text_embedding_cache=TEXT_EMBEDDING_CACHE,
    batch_size=BATCH_SIZE,
    dice_pct=DICE_PCT,
    dice_sensitivity_pcts=DICE_SENSITIVITY_PCTS,
    skip_spin=SKIP_SPIN,
    spin_test_n_perm=1000,
    random_baseline_n=25,
    random_baseline_max_voxels=10000,
    random_seed=13,
    include_voxel_auroc=not SKIP_VOXEL_AUROC,
    sensitivity_jobs=1,
)

result

## Inspect Outputs

In [ ]:
summary = pd.read_csv(result["summary_path"])
summary

In [ ]:
metric_cols = [
    "dataset",
    "requested_model_id",
    "model_id",
    "model_family",
    "status",
    "n_samples",
    "n_ok",
    "pearson_r_mean",
    "spearman_rho_mean",
    "dice_pct90_mean",
    "generated_image_text_normalized_auc_mean",
    "mse_mean",
    "foreground_mse_mean",
    "spatial_corr_mean",
    "checkpoint",
    "skip_reasons",
]
available = [col for col in metric_cols if col in summary.columns]
summary[available] if available else summary

## Visualize Results

Same fixed model-family colors as the other two comparison notebooks. The
top row compares dataset/model means from the summary table; the box plots
below show the full per-sample Pearson r distribution (median/IQR are more
informative than a mean for a noisy per-sample metric), and the last figure
shows Dice overlap sensitivity across percentile thresholds instead of a
single fixed cutoff.

In [ ]:
import matplotlib.pyplot as plt
import plotting_utils as pu

pu.coverage_table(summary)

In [ ]:
fig_metrics, axes = plt.subplots(1, 4, figsize=(21, 4.5))
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="pearson_minus_random_mean",
    ax=axes[0], title="Pearson r minus random baseline (higher is better)", ylabel="Δ Pearson r", show_legend=False,
)
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="dice_pct90_mean",
    ax=axes[1], title="Dice overlap @ top 10% (higher is better)", ylabel="Dice", show_legend=False,
)
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="voxel_auroc_mean",
    ax=axes[2], title="Voxel AUROC (higher is better)", ylabel="AUROC", show_legend=False,
)
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="generated_image_text_normalized_auc_mean",
    ax=axes[3], title="Generated-image/text retrieval AUC", ylabel="Normalized AUC",
)
fig_metrics.suptitle("Text-to-Brain Generation Quality by Dataset and Model", x=0.01, ha="left", fontsize=12)
fig_metrics.tight_layout(rect=(0, 0, 0.88, 0.95))

In [ ]:
by_sample = pd.read_csv(result["by_sample_path"])
by_sample.head(20)

In [ ]:
random_path = result["random_baseline_path"]
random_baseline = pd.read_csv(random_path) if random_path.exists() and random_path.stat().st_size else pd.DataFrame()
random_baseline.head(20)

In [ ]:
dice_path = result["dice_sensitivity_path"]
dice_sensitivity = pd.read_csv(dice_path) if dice_path.exists() and dice_path.stat().st_size else pd.DataFrame()
dice_sensitivity.head(20)

### Per-Sample Distributions And Threshold Sensitivity

In [ ]:
fig_dist = pu.sample_distribution_box(
    by_sample, panel_col="dataset", series_col="model_id", value_col="pearson_r",
    suptitle="Per-Sample Pearson r Distribution by Dataset", ylabel="Pearson r",
)

In [ ]:
fig_dice = pu.small_multiples_lines(
    dice_sensitivity, panel_col="dataset", x_col="pct", y_col="dice", series_col="model_id",
    suptitle="Dice Overlap Sensitivity by Threshold Percentile", xlabel="Percentile threshold", ylabel="Dice",
    aggregate="mean", show_range=True,
)

## Existing Output Loader

Use this cell when you have already run the CLI or a previous notebook cell and only want to inspect the saved CSVs.

In [ ]:
saved_by_sample = OUTPUT_DIR / BY_SAMPLE_FILENAME
saved_summary = OUTPUT_DIR / SUMMARY_FILENAME
saved_random = OUTPUT_DIR / RANDOM_BASELINE_FILENAME
saved_dice = OUTPUT_DIR / DICE_SENSITIVITY_FILENAME

loaded_by_sample = pd.read_csv(saved_by_sample) if saved_by_sample.exists() and saved_by_sample.stat().st_size else pd.DataFrame()
loaded_summary = pd.read_csv(saved_summary) if saved_summary.exists() and saved_summary.stat().st_size else pd.DataFrame()
loaded_random = pd.read_csv(saved_random) if saved_random.exists() and saved_random.stat().st_size else pd.DataFrame()
loaded_dice = pd.read_csv(saved_dice) if saved_dice.exists() and saved_dice.stat().st_size else pd.DataFrame()

loaded_summary.head(20)

## Export For Report

Save every figure and dataframe from this run into one directory (PNG +
CSV, plus a `manifest.json` listing them), so an HTML-report builder can
read a single, predictable location instead of re-running the comparison.

In [ ]:
REPORT_ASSETS_DIR = OUTPUT_DIR / "report_assets" / "text_to_brain_generation"

report_manifest = pu.save_report_assets(
    REPORT_ASSETS_DIR,
    figures={
        "text_to_brain_metrics": fig_metrics,
        "text_to_brain_pearson_distribution": fig_dist,
        "text_to_brain_dice_sensitivity": fig_dice,
    },
    dataframes={
        "text_to_brain_summary": summary,
        "text_to_brain_by_sample": by_sample,
        "text_to_brain_random_baseline": random_baseline,
        "text_to_brain_dice_sensitivity": dice_sensitivity,
    },
)
report_manifest